# Graph Boolean Operations

This notebook demonstrates boolean operations on graphs using topologic_fast.

Graph boolean operations combine two graphs based on their vertices and edges:
- **Union**: Combines all vertices and edges from both graphs
- **Intersection**: Keeps only vertices/edges common to both graphs
- **Difference**: Removes vertices/edges of one graph from another
- **Symmetric Difference**: Keeps vertices/edges in either graph but not both

**Note**: This notebook uses topologic_fast's native `Graph.Union()`, `Graph.Intersect()`, `Graph.Difference()`, and `Graph.SymmetricDifference()` implementations.

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 1. Create Two Overlapping Graphs

We'll create two graphs that share some vertices at similar positions, simulating graphs that can be combined through boolean operations.

In [ ]:
# Create vertices for Graph 1 (a square with center)
v1_0 = tf.Vertex.ByCoordinates(0, 0, 0)   # center
v1_1 = tf.Vertex.ByCoordinates(-2, -2, 0) # bottom-left
v1_2 = tf.Vertex.ByCoordinates(2, -2, 0)  # bottom-right
v1_3 = tf.Vertex.ByCoordinates(2, 2, 0)   # top-right
v1_4 = tf.Vertex.ByCoordinates(-2, 2, 0)  # top-left

# Create edges for Graph 1 (star pattern from center)
e1_0 = tf.Edge.ByStartVertexEndVertex(v1_0, v1_1)
e1_1 = tf.Edge.ByStartVertexEndVertex(v1_0, v1_2)
e1_2 = tf.Edge.ByStartVertexEndVertex(v1_0, v1_3)
e1_3 = tf.Edge.ByStartVertexEndVertex(v1_0, v1_4)
e1_4 = tf.Edge.ByStartVertexEndVertex(v1_1, v1_2)  # bottom edge
e1_5 = tf.Edge.ByStartVertexEndVertex(v1_3, v1_4)  # top edge

# Create Graph 1
g1 = tf.Graph.ByVerticesEdges(
    [v1_0, v1_1, v1_2, v1_3, v1_4],
    [e1_0, e1_1, e1_2, e1_3, e1_4, e1_5]
)

print(f"Graph 1: {g1.Order()} vertices, {g1.Size()} edges")

In [ ]:
# Create vertices for Graph 2 (overlapping triangle)
# Some vertices share positions with Graph 1
v2_0 = tf.Vertex.ByCoordinates(2, 2, 0)   # same as v1_3 (top-right)
v2_1 = tf.Vertex.ByCoordinates(2, -2, 0)  # same as v1_2 (bottom-right)
v2_2 = tf.Vertex.ByCoordinates(5, 0, 0)   # new vertex
v2_3 = tf.Vertex.ByCoordinates(4, 2, 0)   # new vertex
v2_4 = tf.Vertex.ByCoordinates(4, -2, 0)  # new vertex

# Create edges for Graph 2
e2_0 = tf.Edge.ByStartVertexEndVertex(v2_0, v2_1)  # right edge of overlap
e2_1 = tf.Edge.ByStartVertexEndVertex(v2_0, v2_2)  # to apex
e2_2 = tf.Edge.ByStartVertexEndVertex(v2_1, v2_2)  # to apex
e2_3 = tf.Edge.ByStartVertexEndVertex(v2_0, v2_3)  # top connection
e2_4 = tf.Edge.ByStartVertexEndVertex(v2_1, v2_4)  # bottom connection
e2_5 = tf.Edge.ByStartVertexEndVertex(v2_2, v2_3)  # to apex
e2_6 = tf.Edge.ByStartVertexEndVertex(v2_2, v2_4)  # to apex

# Create Graph 2
g2 = tf.Graph.ByVerticesEdges(
    [v2_0, v2_1, v2_2, v2_3, v2_4],
    [e2_0, e2_1, e2_2, e2_3, e2_4, e2_5, e2_6]
)

print(f"Graph 2: {g2.Order()} vertices, {g2.Size()} edges")

## 2. Visualize Both Graphs

In [ ]:
def visualize_graph(graph, name="Graph", color="blue", fig=None, row=1, col=1):
    """Visualize a graph using plotly."""
    if fig is None:
        fig = go.Figure()
    
    # Get vertices and edges
    vertices = graph.Vertices()
    edges = graph.Edges()
    
    # Draw edges
    for edge in edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) == 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            fig.add_trace(go.Scatter(
                x=[p1[0], p2[0], None],
                y=[p1[1], p2[1], None],
                mode='lines',
                line=dict(color=color, width=3),
                showlegend=False,
                hoverinfo='skip'
            ), row=row, col=col)
    
    # Draw vertices
    x_coords = [v.X() for v in vertices]
    y_coords = [v.Y() for v in vertices]
    
    fig.add_trace(go.Scatter(
        x=x_coords,
        y=y_coords,
        mode='markers',
        marker=dict(size=15, color=color, line=dict(color='black', width=2)),
        name=name,
        hovertext=[f"({x:.1f}, {y:.1f})" for x, y in zip(x_coords, y_coords)],
        hoverinfo='text'
    ), row=row, col=col)
    
    return fig

In [ ]:
# Create a figure showing both graphs
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Graph 1 (Blue)', 'Graph 2 (Orange)']
)

visualize_graph(g1, "Graph 1", "blue", fig, row=1, col=1)
visualize_graph(g2, "Graph 2", "orange", fig, row=1, col=2)

fig.update_layout(
    title='Two Overlapping Graphs',
    width=1000,
    height=500
)

fig.update_xaxes(range=[-4, 7], scaleanchor="y", scaleratio=1)
fig.update_yaxes(range=[-4, 4])

fig.show()

In [ ]:
# Show both graphs overlaid
fig_overlay = go.Figure()

# Draw Graph 1
vertices1 = g1.Vertices()
edges1 = g1.Edges()
for edge in edges1:
    edge_verts = edge.Vertices()
    if len(edge_verts) == 2:
        p1 = edge_verts[0].Coordinates()
        p2 = edge_verts[1].Coordinates()
        fig_overlay.add_trace(go.Scatter(
            x=[p1[0], p2[0], None], y=[p1[1], p2[1], None],
            mode='lines', line=dict(color='blue', width=4),
            showlegend=False, hoverinfo='skip'
        ))

fig_overlay.add_trace(go.Scatter(
    x=[v.X() for v in vertices1], y=[v.Y() for v in vertices1],
    mode='markers', marker=dict(size=20, color='blue', line=dict(color='black', width=2)),
    name='Graph 1'
))

# Draw Graph 2
vertices2 = g2.Vertices()
edges2 = g2.Edges()
for edge in edges2:
    edge_verts = edge.Vertices()
    if len(edge_verts) == 2:
        p1 = edge_verts[0].Coordinates()
        p2 = edge_verts[1].Coordinates()
        fig_overlay.add_trace(go.Scatter(
            x=[p1[0], p2[0], None], y=[p1[1], p2[1], None],
            mode='lines', line=dict(color='orange', width=4),
            showlegend=False, hoverinfo='skip'
        ))

fig_overlay.add_trace(go.Scatter(
    x=[v.X() for v in vertices2], y=[v.Y() for v in vertices2],
    mode='markers', marker=dict(size=16, color='orange', line=dict(color='black', width=2)),
    name='Graph 2'
))

fig_overlay.update_layout(
    title='Overlaid Graphs - Notice shared vertex positions at (2,2) and (2,-2)',
    xaxis=dict(title='X', scaleanchor='y', scaleratio=1, range=[-4, 7]),
    yaxis=dict(title='Y', range=[-4, 4]),
    width=800, height=500
)

fig_overlay.show()

## 3. Graph Boolean Operations

topologic_fast provides native boolean operations on graphs: `Union`, `Intersect`, `Difference`, and `SymmetricDifference`.

In [ ]:
def vertices_match(v1, v2, tolerance=0.001):
    """Check if two vertices are at the same position."""
    c1 = v1.Coordinates()
    c2 = v2.Coordinates()
    return (abs(c1[0] - c2[0]) < tolerance and
            abs(c1[1] - c2[1]) < tolerance and
            abs(c1[2] - c2[2]) < tolerance)

def edges_match(e1, e2, tolerance=0.001):
    """Check if two edges connect the same positions."""
    verts1 = e1.Vertices()
    verts2 = e2.Vertices()
    if len(verts1) != 2 or len(verts2) != 2:
        return False
    # Check both orderings
    return ((vertices_match(verts1[0], verts2[0], tolerance) and 
             vertices_match(verts1[1], verts2[1], tolerance)) or
            (vertices_match(verts1[0], verts2[1], tolerance) and 
             vertices_match(verts1[1], verts2[0], tolerance)))

In [ ]:
# Compute union using topologic_fast's native implementation
# Union combines all vertices and edges from both graphs
tolerance = 0.001
union_graph = tf.Graph.Union(g1, g2, tolerance)
print(f"Union Graph: {union_graph.Order()} vertices, {union_graph.Size()} edges")

In [ ]:
# Compute intersection using topologic_fast's native implementation
# Intersection keeps only vertices/edges common to both graphs
intersection_graph = tf.Graph.Intersect(g1, g2, tolerance)
if intersection_graph:
    print(f"Intersection Graph: {intersection_graph.Order()} vertices, {intersection_graph.Size()} edges")
else:
    print("Intersection Graph: Empty (no common vertices)")

In [ ]:
# Compute differences using topologic_fast's native implementation
# Difference removes vertices/edges of second graph from first
diff_ab = tf.Graph.Difference(g1, g2, tolerance)
diff_ba = tf.Graph.Difference(g2, g1, tolerance)

if diff_ab:
    print(f"Difference (A-B): {diff_ab.Order()} vertices, {diff_ab.Size()} edges")
else:
    print("Difference (A-B): Empty")
    
if diff_ba:
    print(f"Difference (B-A): {diff_ba.Order()} vertices, {diff_ba.Size()} edges")
else:
    print("Difference (B-A): Empty")

# Symmetric difference - keeps vertices/edges in either graph but not both
sym_diff = tf.Graph.SymmetricDifference(g1, g2, tolerance)
if sym_diff:
    print(f"Symmetric Difference: {sym_diff.Order()} vertices, {sym_diff.Size()} edges")
else:
    print("Symmetric Difference: Empty")

## 4. Visualize Boolean Results

In [ ]:
def plot_graph_simple(graph, title, color, ax_range=None):
    """Create a simple graph plot."""
    fig = go.Figure()
    
    if graph is None:
        fig.add_annotation(
            x=0.5, y=0.5, text="Empty Graph",
            showarrow=False, font=dict(size=20)
        )
    else:
        vertices = graph.Vertices()
        edges = graph.Edges()
        
        # Draw edges
        for edge in edges:
            edge_verts = edge.Vertices()
            if len(edge_verts) == 2:
                p1 = edge_verts[0].Coordinates()
                p2 = edge_verts[1].Coordinates()
                fig.add_trace(go.Scatter(
                    x=[p1[0], p2[0]], y=[p1[1], p2[1]],
                    mode='lines',
                    line=dict(color=color, width=4),
                    showlegend=False
                ))
        
        # Draw vertices
        fig.add_trace(go.Scatter(
            x=[v.X() for v in vertices],
            y=[v.Y() for v in vertices],
            mode='markers',
            marker=dict(size=18, color=color, line=dict(color='black', width=2)),
            showlegend=False
        ))
    
    if ax_range:
        fig.update_xaxes(range=ax_range[0], scaleanchor='y', scaleratio=1)
        fig.update_yaxes(range=ax_range[1])
    
    fig.update_layout(title=title, width=400, height=400)
    return fig

In [ ]:
# Create subplot figure for all results
fig_results = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        f'Union ({union_graph.Order()} V, {union_graph.Size()} E)',
        f'Intersection ({intersection_graph.Order() if intersection_graph else 0} V, {intersection_graph.Size() if intersection_graph else 0} E)',
        f'Difference A-B ({diff_ab.Order() if diff_ab else 0} V, {diff_ab.Size() if diff_ab else 0} E)',
        f'Difference B-A ({diff_ba.Order() if diff_ba else 0} V, {diff_ba.Size() if diff_ba else 0} E)'
    ]
)

graphs_to_plot = [
    (union_graph, 'green', 1, 1),
    (intersection_graph, 'purple', 1, 2),
    (diff_ab, 'blue', 2, 1),
    (diff_ba, 'orange', 2, 2)
]

for graph, color, row, col in graphs_to_plot:
    if graph is not None:
        vertices = graph.Vertices()
        edges = graph.Edges()
        
        for edge in edges:
            edge_verts = edge.Vertices()
            if len(edge_verts) == 2:
                p1 = edge_verts[0].Coordinates()
                p2 = edge_verts[1].Coordinates()
                fig_results.add_trace(go.Scatter(
                    x=[p1[0], p2[0]], y=[p1[1], p2[1]],
                    mode='lines',
                    line=dict(color=color, width=3),
                    showlegend=False
                ), row=row, col=col)
        
        fig_results.add_trace(go.Scatter(
            x=[v.X() for v in vertices],
            y=[v.Y() for v in vertices],
            mode='markers',
            marker=dict(size=12, color=color, line=dict(color='black', width=2)),
            showlegend=False
        ), row=row, col=col)

fig_results.update_xaxes(range=[-4, 7])
fig_results.update_yaxes(range=[-4, 4])
fig_results.update_layout(
    title='Graph Boolean Operations',
    height=700,
    width=900
)

fig_results.show()

## 5. Graph Metrics Comparison

In [ ]:
def print_graph_metrics(graph, name):
    """Print various graph metrics."""
    if graph is None:
        print(f"\n{name}: Empty Graph")
        return
    
    print(f"\n{name}:")
    print(f"  Order (vertices): {graph.Order()}")
    print(f"  Size (edges):     {graph.Size()}")
    print(f"  Density:          {graph.Density():.3f}")
    print(f"  Diameter:         {graph.Diameter()}")
    print(f"  Max degree:       {graph.MaximumDelta()}")
    print(f"  Min degree:       {graph.MinimumDelta()}")
    print(f"  Is Bipartite:     {graph.IsBipartite()}")
    print(f"  Is Complete:      {graph.IsComplete()}")

print("=" * 50)
print("Graph Metrics Comparison")
print("=" * 50)

print_graph_metrics(g1, "Graph 1 (Original)")
print_graph_metrics(g2, "Graph 2 (Original)")
print_graph_metrics(union_graph, "Union (A + B)")
print_graph_metrics(intersection_graph, "Intersection (A & B)")
print_graph_metrics(diff_ab, "Difference (A - B)")
print_graph_metrics(diff_ba, "Difference (B - A)")

## 6. Create Graphs from CellComplex and Combine

A more practical example: creating graphs from two CellComplexes and merging them.

In [ ]:
# Create two adjacent building sections
building1 = tf.CellComplex.ByCells([
    tf.Cell.Box(0, 0, 0, 2, 2, 1),
    tf.Cell.Box(2, 0, 0, 2, 2, 1)
])

building2 = tf.CellComplex.ByCells([
    tf.Cell.Box(4, 0, 0, 2, 2, 1),
    tf.Cell.Box(4, 2, 0, 2, 2, 1)
])

# Create dual graphs
graph_b1 = tf.Graph.ByTopology(building1)
graph_b2 = tf.Graph.ByTopology(building2)

print(f"Building 1 graph: {graph_b1.Order()} rooms, {graph_b1.Size()} connections")
print(f"Building 2 graph: {graph_b2.Order()} rooms, {graph_b2.Size()} connections")

# Combine the graphs using tf.Graph.Union (they don't share vertices, so union adds them all)
combined_graph = tf.Graph.Union(graph_b1, graph_b2, tolerance)
print(f"\nCombined graph: {combined_graph.Order()} rooms, {combined_graph.Size()} connections")

In [ ]:
# Visualize the buildings and their graphs
fig_buildings = go.Figure()

def draw_cells_2d(cellcomplex, color, fig):
    """Draw a 2D floor plan of cells."""
    cells = cellcomplex.Cells()
    for cell in cells:
        faces = cell.Faces()
        for face in faces:
            vertices = face.Vertices()
            coords = [v.Coordinates() for v in vertices]
            z_coords = [c[2] for c in coords]
            if all(abs(z) < 0.01 for z in z_coords):
                x = [c[0] for c in coords] + [coords[0][0]]
                y = [c[1] for c in coords] + [coords[0][1]]
                fig.add_trace(go.Scatter(
                    x=x, y=y,
                    fill='toself',
                    fillcolor=color,
                    line=dict(color='black', width=2),
                    showlegend=False
                ))
                break

draw_cells_2d(building1, 'rgba(100, 149, 237, 0.5)', fig_buildings)
draw_cells_2d(building2, 'rgba(255, 165, 0, 0.5)', fig_buildings)

# Draw combined graph
vertices = combined_graph.Vertices()
edges = combined_graph.Edges()

for edge in edges:
    edge_verts = edge.Vertices()
    if len(edge_verts) == 2:
        p1 = edge_verts[0].Coordinates()
        p2 = edge_verts[1].Coordinates()
        fig_buildings.add_trace(go.Scatter(
            x=[p1[0], p2[0]], y=[p1[1], p2[1]],
            mode='lines',
            line=dict(color='red', width=4),
            showlegend=False
        ))

fig_buildings.add_trace(go.Scatter(
    x=[v.X() for v in vertices],
    y=[v.Y() for v in vertices],
    mode='markers',
    marker=dict(size=15, color='red'),
    name='Graph Nodes'
))

fig_buildings.update_layout(
    title='Two Building Sections with Combined Connectivity Graph',
    xaxis=dict(title='X', scaleanchor='y', scaleratio=1),
    yaxis=dict(title='Y'),
    width=700, height=500
)

fig_buildings.show()

## Summary

This notebook demonstrated graph boolean operations using topologic_fast:

1. **Union**: Combines all vertices and edges from both graphs, merging vertices at the same position
2. **Intersection**: Keeps only vertices and edges common to both graphs
3. **Difference**: Removes elements of one graph from another
4. **Symmetric Difference**: Keeps elements that exist in exactly one graph

### Key Points

- Graph boolean operations are based on **vertex position matching** with a tolerance
- **Union** is useful for combining connectivity graphs from multiple building sections
- **Intersection** finds shared connectivity patterns between graphs
- **Difference** isolates unique elements in one graph vs another

### topologic_fast Methods Used

- `tf.Graph.Union(graph1, graph2, tolerance)` - Compute union of two graphs
- `tf.Graph.Intersect(graph1, graph2, tolerance)` - Compute intersection of two graphs
- `tf.Graph.Difference(graph1, graph2, tolerance)` - Compute difference (A - B)
- `tf.Graph.SymmetricDifference(graph1, graph2, tolerance)` - Compute symmetric difference
- `tf.Vertex.ByCoordinates()` - Create vertices
- `tf.Edge.ByStartVertexEndVertex()` - Create edges
- `tf.Graph.ByVerticesEdges()` - Create graphs
- `tf.Graph.ByTopology()` - Create dual graphs from topology
- `graph.Order()`, `graph.Size()`, `graph.Density()`, etc. - Graph metrics